# A Simple CNN for Sentence Classification

In this notebook, we train a small convolutional neural network (CNN) to classify movie-review sentences as positive or negative. We will load the data, turn words into numbers, build and train a model, and check its accuracy. A CNN looks for patterns in short groups of neighboring words. The goal is to understand the basic steps, not reproduce a research paper.

## 1. Load the data

Each sentence has a label: `1` means positive and `0` means negative.

In [1]:
import random
from collections import Counter
import nltk
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

nltk.download("sentence_polarity", quiet=True)
from nltk.corpus import sentence_polarity

random.seed(42)
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

data = [(words, 1) for words in sentence_polarity.sents(categories="pos")]
data += [(words, 0) for words in sentence_polarity.sents(categories="neg")]
random.shuffle(data)
split = int(0.8 * len(data))
train_data, test_data = data[:split], data[split:]
print("Training examples:", len(train_data))
print("Test examples:", len(test_data))
print("Example:", " ".join(train_data[0][0]), "->", train_data[0][1])

Using: cuda
Training examples: 8529
Test examples: 2133
Example: chicago is , in many ways , an admirable achievement . -> 1


## 2. Turn words into numbers

We give each training word an ID. `0` is padding and `1` means an unknown word. Each sentence is padded or shortened to 40 words.

In [11]:
word_counts = Counter(word for sentence, _ in train_data for word in sentence)
vocabulary = {"<pad>": 0, "<unknown>": 1}
for word in word_counts:
    vocabulary[word] = len(vocabulary)

word_counts

Counter({'.': 11240,
         'the': 8023,
         ',': 7986,
         'a': 5849,
         'and': 4975,
         'of': 4859,
         'to': 3399,
         'is': 2686,
         'in': 2061,
         'that': 1955,
         'it': 1810,
         'as': 1442,
         'but': 1316,
         'with': 1251,
         'film': 1188,
         'this': 1156,
         'for': 1130,
         'its': 1077,
         'an': 1053,
         'movie': 1018,
         "it's": 893,
         'be': 757,
         'you': 718,
         'on': 711,
         'not': 641,
         'by': 616,
         'about': 610,
         'more': 586,
         'like': 585,
         'are': 575,
         'one': 575,
         'has': 566,
         'at': 562,
         'from': 538,
         '"': 533,
         'than': 533,
         'all': 513,
         '--': 505,
         'his': 504,
         'have': 496,
         'so': 443,
         'if': 436,
         'or': 411,
         'story': 374,
         'i': 371,
         'too': 358,
         'just': 355,


In [12]:
vocabulary

{'<pad>': 0,
 '<unknown>': 1,
 'chicago': 2,
 'is': 3,
 ',': 4,
 'in': 5,
 'many': 6,
 'ways': 7,
 'an': 8,
 'admirable': 9,
 'achievement': 10,
 '.': 11,
 'the': 12,
 'last': 13,
 '20': 14,
 'minutes': 15,
 'are': 16,
 'somewhat': 17,
 'redeeming': 18,
 'but': 19,
 'most': 20,
 'of': 21,
 'movie': 22,
 'same': 23,
 'teenage': 24,
 'american': 25,
 'road-trip': 26,
 'drek': 27,
 "we've": 28,
 'seen': 29,
 'before': 30,
 '-': 31,
 'only': 32,
 'this': 33,
 'time': 34,
 'you': 35,
 'have': 36,
 'to': 37,
 'read': 38,
 'fart': 39,
 'jokes': 40,
 'sheer': 41,
 'joy': 42,
 'and': 43,
 'pride': 44,
 'they': 45,
 'took': 46,
 'their': 47,
 'work': 48,
 '--': 49,
 'each': 50,
 'other': 51,
 'shines': 52,
 'through': 53,
 'every': 54,
 'frame': 55,
 "it's": 56,
 'easy': 57,
 'love': 58,
 'robin': 59,
 'tunney': 60,
 "she's": 61,
 'pretty': 62,
 'she': 63,
 'can': 64,
 'act': 65,
 'it': 66,
 'gets': 67,
 'harder': 68,
 'understand': 69,
 'her': 70,
 'choices': 71,
 'his': 72,
 'role': 73,
 'obse

In [5]:
max_length = 40
def encode(sentence):
    ids = [vocabulary.get(word, 1) for word in sentence[:max_length]]
    return ids + [0] * (max_length - len(ids))

X_train = torch.tensor([encode(sentence) for sentence, _ in train_data])
y_train = torch.tensor([label for _, label in train_data])
X_test = torch.tensor([encode(sentence) for sentence, _ in test_data])
y_test = torch.tensor([label for _, label in test_data])
train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=64, shuffle=True)
test_loader = DataLoader(TensorDataset(X_test, y_test), batch_size=64)
print("Vocabulary size:", len(vocabulary))
print("First 12 word IDs:", X_train[0][:12].tolist())

Vocabulary size: 19059
First 12 word IDs: [2, 3, 4, 5, 6, 7, 4, 8, 9, 10, 11, 0]


In [9]:
train_data[0],  X_train[0], len(train_data[0]), len(X_train[0])

((['chicago',
   'is',
   ',',
   'in',
   'many',
   'ways',
   ',',
   'an',
   'admirable',
   'achievement',
   '.'],
  1),
 tensor([ 2,  3,  4,  5,  6,  7,  4,  8,  9, 10, 11,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,  0,
          0,  0,  0,  0]),
 2,
 40)

## 3. Build the CNN

The embedding layer turns each word ID into a vector. A convolution looks at groups of three neighboring words. Max pooling keeps the strongest pattern, and the final layer predicts the label.

In [10]:
class SentenceCNN(nn.Module):
    def __init__(self, vocabulary_size, embedding_size=50):
        super().__init__()
        self.embedding = nn.Embedding(vocabulary_size, embedding_size, padding_idx=0)
        self.convolution = nn.Conv1d(embedding_size, 64, kernel_size=3)
        self.classifier = nn.Linear(64, 2)

    def forward(self, word_ids):
        vectors = self.embedding(word_ids).transpose(1, 2)
        features = F.relu(self.convolution(vectors))
        strongest = features.max(dim=2).values
        return self.classifier(strongest)
        
model = SentenceCNN(len(vocabulary)).to(device)
print(model)

SentenceCNN(
  (embedding): Embedding(19059, 50, padding_idx=0)
  (convolution): Conv1d(50, 64, kernel_size=(3,), stride=(1,))
  (classifier): Linear(in_features=64, out_features=2, bias=True)
)


## 4. Train the model

For each batch, the model makes predictions, measures its mistakes, and updates its weights. Try changing the number of epochs.

In [13]:
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
epochs = 5
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for word_ids, labels in train_loader:
        word_ids, labels = word_ids.to(device), labels.to(device)
        predictions = model(word_ids)
        loss = F.cross_entropy(predictions, labels)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch + 1}: loss = {total_loss / len(train_loader):.3f}")

Epoch 1: loss = 0.682
Epoch 2: loss = 0.612
Epoch 3: loss = 0.539
Epoch 4: loss = 0.447
Epoch 5: loss = 0.352


## 5. Check accuracy

The test sentences were not used for training. Accuracy is the fraction labeled correctly.

In [14]:
model.eval()
correct = total = 0
with torch.no_grad():
    for word_ids, labels in test_loader:
        word_ids, labels = word_ids.to(device), labels.to(device)
        predictions = model(word_ids).argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)
print(f"Test accuracy: {correct / total:.1%}")

Test accuracy: 66.9%


## Your turn

1. Change `epochs` from 5 to 2. How do the loss and accuracy change?
2. Change `kernel_size` from 3 to 5. What might a filter over five words learn?
3. Explain why sentences are padded with zeros.

Accuracy can vary; focus on understanding what each part of the model does.